# Битва при Ватерлоо — построение RDF-графа

Файлы `ontology.ttl` и `dataset.json` загружаются в корень диска вручную.


In [ ]:
!pip install -q rdflib

In [ ]:
import json
from pathlib import Path

from rdflib import Graph, Literal, Namespace
from rdflib.namespace import RDF, RDFS

ROOT = Path("/content")
EX = Namespace("http://itmo.ru/waterloo#")

g = Graph()
g.parse(ROOT / "ontology.ttl", format="turtle")
g.bind("ex", EX)

with open(ROOT / "dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

battle_uri = EX["BattleOfWaterloo"]
g.add((battle_uri, RDF.type, EX.Battle))
g.add((battle_uri, RDFS.label, Literal("Битва при Ватерлоо")))

for sent in data.get("sentences", []):
    sent_uri = EX[f"sentence_{sent["id"]}"]
    g.add((sent_uri, RDF.type, EX.Sentence))
    g.add((sent_uri, EX.hasTopic, Literal(sent.get("topic", ""))))
    g.add((sent_uri, RDFS.label, Literal(sent.get("text", ""))))

    for i, ctx_text in enumerate(sent.get("context", [])):
        ctx_uri = EX[f"sentence_{sent["id"]}_context_{i}"]
        g.add((ctx_uri, RDF.type, EX.Sentence))
        g.add((ctx_uri, RDFS.label, Literal(ctx_text)))
        g.add((sent_uri, EX.hasContext, ctx_uri))

    for j, atomic in enumerate(sent.get("atomic_statements", [])):
        atomic_uri = EX[f"sentence_{sent["id"]}_atomic_{j}"]
        g.add((atomic_uri, RDF.type, EX.AtomicStatement))
        g.add((atomic_uri, RDFS.label, Literal(atomic.get("statement", ""))))
        g.add((sent_uri, EX.hasAtomicStatement, atomic_uri))

        for obj in atomic.get("objects", []):
            obj_uri = EX[f"object_{obj.replace(" ", "_")}"]
            g.add((obj_uri, RDF.type, EX.Object))
            g.add((obj_uri, RDFS.label, Literal(obj)))
            g.add((atomic_uri, EX.mentionsObject, obj_uri))

Path("output").mkdir(exist_ok=True)
g.serialize(destination="output/knowledge_graph.ttl", format="turtle")
print(f"Граф создан: {len(g)} триплетов")

In [ ]:
from google.colab import files
files.download("output/knowledge_graph.ttl")